In [ ]:
# Load the precomputed embeddings and metadata.
# Run LLM-based retrieval, similarity search, or chatbot logic.
# Compare different models or approaches using the saved embeddings.

#### Goal: Test out the LLM by feeding the normalized data as instructions/additional info

In [18]:
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import pickle

##### 1. Test the ```sentence-transformers/all-MiniLM-L6-v2``` model

In [70]:
# Load data
dataset_miniLM_path = "../data/RDS_2019_short_MiniLM_L6_v2.pkl"

with open(dataset_miniLM_path, "rb") as f:
    dataset_miniLM = pickle.load(f)

# Load model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [71]:
# Question
question = "Which rule sets chilled water supply temperature to 44F?" # expect rule 22-1
#question ="Which rule sets baseline heat-rejection device to have a design temperature rise of 10F?" # expect rule 22-14
#question = "What are the rules for the baseline heat rejection device?"

# Encode the question
question_embedding = model.encode(question, normalize_embeddings=True)

rule_embeddings = torch.tensor([r["embedding"] for r in dataset_miniLM])
cos_scores = util.cos_sim(question_embedding, rule_embeddings)[0]

# Find the top k most similar rules
k = 3  # Set the number of top results you want
top_k_indices = torch.topk(cos_scores, k=min(k, len(dataset_miniLM))).indices

print("\n")
print(f"Top {k} most relevant rules:")
print("=" * 80)
for i, idx in enumerate(top_k_indices, 1):
    top_rule = dataset_miniLM[idx]
    print(f"\n{i}. Rule ID: {top_rule['rule_id']}")
    print(f"   Rule description: {top_rule['rule_description']}")
    print(f"   Similarity score: {cos_scores[idx].item():.4f}")



Top 3 most relevant rules:

1. Rule ID: 22-1
   Rule description: Baseline chilled water design supply temperature shall be modeled at 44F.
   Similarity score: 0.8621

2. Rule ID: 22-2
   Rule description: Baseline chilled water design return temperature shall be modeled at 56F.
   Similarity score: 0.7198

3. Rule ID: 22-4
   Rule description: For Baseline chilled water loop that is not purchased chilled water and does not serve any computer room HVAC systems, chilled-water supply temperature shall be reset using the following schedule: 44F at outdoor dry-bulb temperature of 80F and above, 54F at 60F and below, and ramped linearly between 44F and 54F at temperature between 80F and 60F.
   Similarity score: 0.6639


##### 2 Test the ```intfloat/e5-base-v2``` model

In [78]:
# Load data
dataset_e5_base_path = "../data/RDS_2019_short_e5_base_v2.pkl"

with open(dataset_e5_base_path, "rb") as f:
    dataset_e5_base = pickle.load(f)

# Load model
model = SentenceTransformer("intfloat/e5-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [79]:
# question
question = "Which rule sets chilled water supply temperature to 44F?" # expect rule 22-1
#question ="Which rule sets baseline heat-rejection device to have a design temperature rise of 10F?" # expect rule 22-14
#question = "What are the rules for the baseline heat rejection device?"

# Encode the question
question_embedding = model.encode(question, normalize_embeddings=True)

# Compute cosine similarity
rule_embeddings = torch.tensor([r["embedding"] for r in dataset_e5_base])
cos_scores = util.cos_sim(question_embedding, rule_embeddings)[0]

# Find the top k most similar rules
k = 3  # Set the number of top results you want
top_k_indices = torch.topk(cos_scores, k=min(k, len(dataset_e5_base))).indices

print("\n")
print(f"Top {k} most relevant rules:")
print("=" * 80)
for i, idx in enumerate(top_k_indices, 1):
    top_rule = dataset_e5_base[idx]
    print(f"\n{i}. Rule ID: {top_rule['rule_id']}")
    print(f"   Rule description: {top_rule['rule_description']}")
    print(f"   Similarity score: {cos_scores[idx].item():.4f}")



Top 3 most relevant rules:

1. Rule ID: 22-1
   Rule description: Baseline chilled water design supply temperature shall be modeled at 44F.
   Similarity score: 0.9032

2. Rule ID: 22-4
   Rule description: For Baseline chilled water loop that is not purchased chilled water and does not serve any computer room HVAC systems, chilled-water supply temperature shall be reset using the following schedule: 44F at outdoor dry-bulb temperature of 80F and above, 54F at 60F and below, and ramped linearly between 44F and 54F at temperature between 80F and 60F.
   Similarity score: 0.8733

3. Rule ID: 22-6
   Rule description: For Baseline chilled water loop that is not purchased chilled water and serves computer room HVAC systems (System Type-11), The maximum reset chilled-water supply temperature shall be 54F.
   Similarity score: 0.8691


##### 3. Summarization: Test the ```facebook/bart-large-cnn``` model

In [55]:
import json

dataset_path = "../data/RDS_2019_short.jsonl"
dataset = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        dataset.append(json.loads(line))

In [56]:
# Find the rule you want to summarize
rule_id_to_find = "22-1"
rule_data = next((item for item in dataset if item["rule_id"] == rule_id_to_find), None)

if rule_data:
    text_to_summarize = rule_data["rule_logic"]
    
    # Tokenize
    inputs = tokenizer(
        text_to_summarize,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )
    
    # Generate summary
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=500,
        min_length=10,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )
    
    # Decode and print
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print(f"Summary for {rule_id_to_find}: {summary}")
else:
    print(f"Rule {rule_id_to_find} not found")

Summary for 22-1: `if any(sys_type in baseline_hvac_system_dict.keys() forsys_ type in ["SYS-7", "Sys-8", " sys-11.1", "sys-12," "sYS-13"), CHECK_RULE_LOGIC): CHECK RULE LOGIC. Else, rule is not applicable to B-RMR: `else: RULE_NOT_APPLICABLE’


In [58]:
text_to_summarize

'`if any(sys_type in baseline_hvac_system_dict.keys() for sys_type in ["SYS-7", "SYS-8", "SYS-11.1", "SYS-11.2", "SYS-12", "SYS-13", "SYS-1A", "SYS-3A", "SYS-7A", "SYS-8A", "SYS-11.1A", "SYS-11.2A", "SYS-12A", "SYS-13A", "SYS-7B", "SYS-8B", "SYS-11B", "SYS-12B", "SYS-1C", "SYS-3C", "SYS-7C", "SYS-11C"]): CHECK_RULE_LOGIC`\n\n  - Else, rule is not applicable to B-RMR: `else: RULE_NOT_APPLICABLE`\n\n## Rule Logic:  \n\n- For each boiler in B_RMI, save boiler to loop boiler dictionary: `loop_boiler_dict = find_all(B-RMI)`\n\n- For each fluid loop in B_RMR: `for fluid_loop_b in B_RMR.ASHRAE229.fluid_loops:`\n\n  - Check if fluid loop is connected to chiller(s): `if fluid_loop_b.id in loop_chiller_dict.keys()`\n\n    **Rule Assertion - Component:**\n\n    - Case 1: For baseline primary chilled water loop, if design supply temperature is 44F: `if fluid_loop_b.cooling_or_condensing_design_and_control.design_supply_temperature == 44: PASS`\n\n    - Case 2: Else: `else: FAIL`\n\n**[Back](../_to

- We tested gpt5.2 to see how it performs compared to the SLM we tried above.

Prompt)
```
I will provide you rule checking logic. You need to summarize it in human readable way and description needs to be concise and explanable to engineers who are in this domain.
<rule_logic> if any(sys_type in baseline_hvac_system_dict.keys() for sys_type in [\"SYS-7\", \"SYS-8\", \"SYS-11.1\", \"SYS-11.2\", \"SYS-12\", \"SYS-13\", \"SYS-1A\", \"SYS-3A\", \"SYS-7A\", \"SYS-8A\", \"SYS-11.1A\", \"SYS-11.2A\", \"SYS-12A\", \"SYS-13A\", \"SYS-7B\", \"SYS-8B\", \"SYS-11B\", \"SYS-12B\", \"SYS-1C\", \"SYS-3C\", \"SYS-7C\", \"SYS-11C\"]): CHECK_RULE_LOGIC\n\n - Else, rule is not applicable to B-RMR: else: RULE_NOT_APPLICABLE\n\n## Rule Logic: \n\n- For each boiler in B_RMI, save boiler to loop boiler dictionary: loop_boiler_dict = find_all(B-RMI)\n\n- For each fluid loop in B_RMR: for fluid_loop_b in B_RMR.ASHRAE229.fluid_loops:\n\n - Check if fluid loop is connected to chiller(s): if fluid_loop_b.id in loop_chiller_dict.keys()\n\n Rule Assertion - Component:\n\n - Case 1: For baseline primary chilled water loop, if design supply temperature is 44F: if fluid_loop_b.cooling_or_condensing_design_and_control.design_supply_temperature == 44: PASS\n\n - Case 2: Else: else: FAIL\n\n** </rule_logic>
```

Answer)   
Rule applicability (B-RMR):
This rule only applies if the baseline HVAC system type includes any of: SYS-7, SYS-8, SYS-11.1, SYS-11.2, SYS-12, SYS-13 and their variants A/B/C (including SYS-1A, SYS-3A, SYS-7A, SYS-8A, SYS-11.1A, SYS-11.2A, SYS-12A, SYS-13A, SYS-7B, SYS-8B, SYS-11B, SYS-12B, SYS-1C, SYS-3C, SYS-7C, SYS-11C). If none are present, the rule is Not Applicable.

What it checks:
For each baseline fluid loop in the B-RMR that is connected to one or more chillers (i.e., a chilled-water loop), verify the loop’s cooling/condensing design supply temperature is 44°F.

Pass/Fail criteria:

PASS if design_supply_temperature == 44°F for each applicable (chiller-connected) baseline loop.
FAIL otherwise.
(Note: The logic mentions collecting boilers from B-RMI, but that data is not used in the shown assertion.)

Prompt2)   
```
I will provide you rule checking logic. You need to summarize it in human readable way and description needs to be concise and explanable to engineers who are in this domain.
<rule_logic> if heat_rejection.loop.id in heat_rejection_loop_ids_b: CONTINUE TO RULE LOGI\n \n - Else, rule is not applicable to the heat rejection: else: RULE_NOT_APPLICABL\n\n ## Rule Logic: \n\n - get the fan speed control: fan_speed_control_type_b = heat_rejection.fan_speed_contro\n\n **Rule Assertion - HeatRejection:**\n\n - Case 1: If the fan speed control equals VARIABLE_SPEED, PASS: if ( fan_speed_control_type_b == VARIABLE_SPEED ): PAS\n\n - Case 2: Else: else: FAIL
```

Answer)   
Rule applicability:
Applies only to a baseline heat rejection component whose loop id is included in the set of baseline heat rejection loop IDs (heat_rejection.loop.id in heat_rejection_loop_ids_b). If not, the rule is Not Applicable for that heat rejection.

What it checks:
For each applicable heat rejection device, check the fan speed control type.

Pass/Fail criteria:

PASS if fan_speed_control == VARIABLE_SPEED.
FAIL otherwise.